# IT4060 - HPC Failure Prediction

## Notebook: Data Exploration

This notebook explores the two LANL datasets used in the project:

- failure logs for label generation
- usage logs for workload feature engineering

The focus here is understanding structure, coverage, quality, and merge readiness before modeling.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)
sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.figsize'] = (14, 6)

In [ ]:
PROJECT_NAME = 'IT4060-ML-Assignment-HPC-Failure-Prediction'
FAILURE_FILE = 'LA-UR-05-7318-failure-data-1996-2005.csv'
USAGE_FILE = 'LA-UR-06-0803-MX20_NODES_0_TO_255_NODE-Z.TXT'
BASE_USAGE_COLUMNS = [
    'job_end_time', 'user_id', 'num_procs', 'submit_time', 'suggested_time',
    'deadline_time', 'dispatch_time', 'cpu_user_time', 'cpu_system_time',
    'total_time', 'batch_id'
]

def find_project_root():
    cwd = Path.cwd().resolve()
    home = Path.home().resolve()
    desktop = home / 'Desktop'
    candidate_roots = [cwd, *cwd.parents, home, desktop, desktop / 'Manilka' / 'ML_Assignment']
    seen = set()

    for base in candidate_roots:
        for candidate in (base, base / PROJECT_NAME):
            if candidate in seen or not candidate.exists():
                continue
            seen.add(candidate)

            failure_path = candidate / 'data' / 'raw' / 'lanl_failure' / FAILURE_FILE
            usage_path = candidate / 'data' / 'raw' / 'lanl_usage' / USAGE_FILE
            if failure_path.exists() and usage_path.exists():
                return candidate

    for search_root in (desktop, home):
        if not search_root.exists():
            continue
        for match in search_root.rglob(FAILURE_FILE):
            candidate = match.parents[3]
            usage_path = candidate / 'data' / 'raw' / 'lanl_usage' / USAGE_FILE
            if usage_path.exists():
                return candidate

    raise FileNotFoundError('Could not locate the project data directory.')

def parse_usage_line(line):
    parts = line.strip().split(maxsplit=12)
    if len(parts) < 11:
        return None

    row = dict(zip(BASE_USAGE_COLUMNS, parts[:11]))
    row['status'] = parts[11] if len(parts) >= 12 else None
    row['nodes_raw'] = parts[12] if len(parts) == 13 else None
    return row

def extract_nodes(nodes_raw):
    if not isinstance(nodes_raw, str):
        return []

    cleaned = nodes_raw.replace('/', ' ').strip()
    if not cleaned:
        return []

    nodes = []
    for token in cleaned.split():
        if '-' in token:
            start, end = token.split('-', 1)
            nodes.extend(range(int(start), int(end) + 1))
        else:
            nodes.append(int(token))
    return sorted(set(nodes))

def first_non_null_label(row, columns):
    for column in columns:
        value = row[column]
        if pd.notna(value):
            return column
    return 'Unknown'

project_root = find_project_root()
failure_path = project_root / 'data' / 'raw' / 'lanl_failure' / FAILURE_FILE
usage_path = project_root / 'data' / 'raw' / 'lanl_usage' / USAGE_FILE
interim_dir = project_root / 'data' / 'interim'
interim_dir.mkdir(parents=True, exist_ok=True)

print(f'Working directory: {Path.cwd()}')
print(f'Project root: {project_root}')
print(f'Failure data path: {failure_path}')
print(f'Usage data path: {usage_path}')
print(f'Interim data path: {interim_dir}')

In [ ]:
failure_df = pd.read_csv(failure_path, skiprows=1).rename(columns={
    'Prob Started (mm/dd/yy hh:mm)': 'failure_start',
    'Prob Fixed (mm/dd/yy hh:mm)': 'failure_end',
})
failure_df['failure_start'] = pd.to_datetime(failure_df['failure_start'], errors='coerce')
failure_df['failure_end'] = pd.to_datetime(failure_df['failure_end'], errors='coerce')
failure_df['failure_category'] = failure_df.apply(
    first_non_null_label,
    axis=1,
    columns=['Facilities', 'Hardware', 'Human Error', 'Network', 'Undetermined', 'Software'],
)

usage_rows = []
skipped_short_lines = 0
with usage_path.open() as f:
    for line_number, line in enumerate(f, start=1):
        parsed = parse_usage_line(line)
        if parsed is None:
            skipped_short_lines += 1
            continue
        parsed['line_number'] = line_number
        usage_rows.append(parsed)

usage_df = pd.DataFrame(usage_rows)
for column in BASE_USAGE_COLUMNS:
    usage_df[column] = pd.to_numeric(usage_df[column], errors='coerce')

usage_df['status'] = usage_df['status'].astype('string').str.strip()
usage_df['usage_time'] = pd.to_datetime(usage_df['job_end_time'], unit='s', errors='coerce')
usage_df['submit_time'] = pd.to_datetime(usage_df['submit_time'], unit='s', errors='coerce')
usage_df['dispatch_time'] = pd.to_datetime(usage_df['dispatch_time'], unit='s', errors='coerce')
usage_df['nodes'] = usage_df['nodes_raw'].apply(extract_nodes)
usage_df['node_count'] = usage_df['nodes'].str.len()
usage_df['has_node_info'] = usage_df['node_count'].gt(0)

usage_df = usage_df.loc[
    usage_df['usage_time'].notna() &
    (usage_df['job_end_time'] >= 1_000_000_000) &
    usage_df['batch_id'].notna()
].copy()

failure_system20 = failure_df.loc[
    (failure_df['System'] == 20) &
    (failure_df['nodenumz'].between(0, 255, inclusive='both')) &
    failure_df['failure_start'].notna()
].copy()
failure_system20['nodenumz'] = failure_system20['nodenumz'].astype(int)

usage_with_nodes = usage_df.loc[usage_df['has_node_info']].explode('nodes').rename(columns={'nodes': 'node'}).copy()
usage_with_nodes['node'] = usage_with_nodes['node'].astype(int)
usage_with_nodes = usage_with_nodes.loc[usage_with_nodes['node'].between(0, 255)].copy()

overview = pd.DataFrame([
    {
        'dataset': 'failure',
        'rows': len(failure_df),
        'columns': failure_df.shape[1],
        'start_time': failure_df['failure_start'].min(),
        'end_time': failure_df['failure_end'].max(),
    },
    {
        'dataset': 'usage',
        'rows': len(usage_df),
        'columns': usage_df.shape[1],
        'start_time': usage_df['usage_time'].min(),
        'end_time': usage_df['usage_time'].max(),
    },
])

display(overview)
print(f'Skipped malformed short usage lines: {skipped_short_lines:,}')

## Persist Interim Data

These cleaned exports are used by the next notebook so we do not need to parse the raw files again.

In [ ]:
failure_export_path = interim_dir / 'failure_system20_nodes_0_255.csv'
usage_jobs_export_path = interim_dir / 'usage_jobs_clean.csv.gz'
usage_nodes_export_path = interim_dir / 'usage_node_events.csv.gz'

failure_export = failure_system20.sort_values(['failure_start', 'nodenumz']).copy()
usage_jobs_export = usage_df.sort_values(['usage_time', 'batch_id']).copy()
usage_jobs_export['nodes'] = usage_jobs_export['nodes'].apply(lambda values: ' '.join(map(str, values)) if values else '')
usage_nodes_export = usage_with_nodes.sort_values(['usage_time', 'node', 'batch_id']).copy()

failure_export.to_csv(failure_export_path, index=False)
usage_jobs_export.to_csv(usage_jobs_export_path, index=False, compression='gzip')
usage_nodes_export.to_csv(usage_nodes_export_path, index=False, compression='gzip')

interim_manifest = pd.DataFrame([
    {'file_name': failure_export_path.name, 'rows': len(failure_export), 'description': 'System 20 failure records filtered to nodes 0-255'},
    {'file_name': usage_jobs_export_path.name, 'rows': len(usage_jobs_export), 'description': 'Cleaned usage job records with parsed timestamps'},
    {'file_name': usage_nodes_export_path.name, 'rows': len(usage_nodes_export), 'description': 'Usage job records exploded to one row per node'},
])

display(interim_manifest)
print('Interim files written successfully.')

## Failure Dataset Exploration

In [ ]:
failure_summary = pd.DataFrame([
    {'metric': 'Total failure records', 'value': len(failure_df)},
    {'metric': 'Unique systems', 'value': failure_df['System'].nunique()},
    {'metric': 'Unique nodes in System 20 (0-255)', 'value': failure_system20['nodenumz'].nunique()},
    {'metric': 'System 20 failure rows on nodes 0-255', 'value': len(failure_system20)},
    {'metric': 'Failure time range start', 'value': failure_df['failure_start'].min()},
    {'metric': 'Failure time range end', 'value': failure_df['failure_end'].max()},
])

display(failure_summary)
display(failure_df[['System', 'machine type', 'nodenum', 'nodenumz', 'failure_start', 'failure_end', 'failure_category', 'Same Event']].head())

failure_missing = failure_df[['Facilities', 'Hardware', 'Human Error', 'Network', 'Undetermined', 'Software']].isna().mean().sort_values(ascending=False).rename('missing_ratio')
display(failure_missing.to_frame())

In [ ]:
top_systems = failure_df['System'].value_counts().head(10).sort_values(ascending=False).rename_axis('System').reset_index(name='failure_count')
category_counts = failure_df['failure_category'].value_counts().head(10).rename_axis('failure_category').reset_index(name='failure_count')
failure_year_counts = failure_df.dropna(subset=['failure_start']).assign(year=failure_df['failure_start'].dt.year).groupby('year').size().reset_index(name='failure_count')
system20_top_nodes = failure_system20['nodenumz'].value_counts().head(15).sort_values(ascending=False).rename_axis('node').reset_index(name='failure_count')

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
sns.barplot(data=top_systems, x='failure_count', y='System', hue='System', dodge=False, legend=False, ax=axes[0, 0], palette='Blues_r')
axes[0, 0].set_title('Top Systems by Failure Count')
axes[0, 0].set_xlabel('Failure records')
axes[0, 0].set_ylabel('System')

sns.barplot(data=category_counts, x='failure_count', y='failure_category', hue='failure_category', dodge=False, legend=False, ax=axes[0, 1], palette='rocket')
axes[0, 1].set_title('Failure Categories')
axes[0, 1].set_xlabel('Failure records')
axes[0, 1].set_ylabel('Category source column')

sns.lineplot(data=failure_year_counts, x='year', y='failure_count', marker='o', ax=axes[1, 0], color='teal')
axes[1, 0].set_title('Failure Records by Year')
axes[1, 0].set_xlabel('Year')
axes[1, 0].set_ylabel('Failure records')

sns.barplot(data=system20_top_nodes, x='failure_count', y='node', hue='node', dodge=False, legend=False, ax=axes[1, 1], palette='viridis')
axes[1, 1].set_title('Most Failure-Prone Nodes in System 20 (0-255)')
axes[1, 1].set_xlabel('Failure records')
axes[1, 1].set_ylabel('Node')

plt.tight_layout()
plt.show()

## Usage Dataset Exploration

In [ ]:
usage_summary = pd.DataFrame([
    {'metric': 'Parsed usage records', 'value': len(usage_df)},
    {'metric': 'Rows with explicit node info', 'value': int(usage_df['has_node_info'].sum())},
    {'metric': 'Unique users', 'value': usage_df['user_id'].nunique()},
    {'metric': 'Unique batch IDs', 'value': usage_df['batch_id'].nunique()},
    {'metric': 'Usage time range start', 'value': usage_df['usage_time'].min()},
    {'metric': 'Usage time range end', 'value': usage_df['usage_time'].max()},
])

display(usage_summary)
display(usage_df[['usage_time', 'user_id', 'num_procs', 'cpu_user_time', 'cpu_system_time', 'total_time', 'status', 'nodes_raw', 'node_count']].head())

usage_status_counts = usage_df['status'].fillna('missing').value_counts().rename_axis('status').reset_index(name='count')
display(usage_status_counts)

usage_numeric_summary = usage_df[['num_procs', 'cpu_user_time', 'cpu_system_time', 'total_time', 'node_count']].describe().T
display(usage_numeric_summary)

In [ ]:
status_counts = usage_df['status'].fillna('missing').value_counts().rename_axis('status').reset_index(name='count')
top_proc_requests = usage_df['num_procs'].value_counts().head(15).rename_axis('num_procs').reset_index(name='job_count')
monthly_usage_jobs = usage_df.set_index('usage_time').resample('ME').size().reset_index(name='job_count')
usage_duration_sample = usage_df.loc[usage_df['total_time'] > 0, 'total_time']
usage_duration_sample = usage_duration_sample.sample(min(20000, len(usage_duration_sample)), random_state=42)
usage_duration_plot = pd.DataFrame({'log10_total_time': np.log10(usage_duration_sample)})
node_count_distribution = usage_df.loc[usage_df['has_node_info'], 'node_count'].clip(upper=64)

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
sns.barplot(data=status_counts, x='count', y='status', hue='status', dodge=False, legend=False, ax=axes[0, 0], palette='mako')
axes[0, 0].set_title('Usage Job Status Distribution')
axes[0, 0].set_xlabel('Jobs')
axes[0, 0].set_ylabel('Status')

sns.barplot(data=top_proc_requests, x='num_procs', y='job_count', hue='num_procs', dodge=False, legend=False, ax=axes[0, 1], palette='crest')
axes[0, 1].set_title('Most Common Requested Processor Counts')
axes[0, 1].set_xlabel('Requested processors')
axes[0, 1].set_ylabel('Jobs')

sns.lineplot(data=monthly_usage_jobs, x='usage_time', y='job_count', ax=axes[1, 0], color='darkorange')
axes[1, 0].set_title('Monthly Usage Job Volume')
axes[1, 0].set_xlabel('Month')
axes[1, 0].set_ylabel('Jobs')

sns.histplot(data=usage_duration_plot, x='log10_total_time', bins=40, ax=axes[1, 1], color='slateblue')
axes[1, 1].set_title('Distribution of Job Duration Proxy')
axes[1, 1].set_xlabel('log10(total_time)')
axes[1, 1].set_ylabel('Jobs')

plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 5))
sns.histplot(node_count_distribution, bins=30, color='seagreen')
plt.title('Node Count per Job for Rows With Node Information')
plt.xlabel('Number of nodes referenced in the job record')
plt.ylabel('Jobs')
plt.tight_layout()
plt.show()

## Overlap Between Failure and Usage Data

In [ ]:
usage_min = usage_df['usage_time'].min()
usage_max = usage_df['usage_time'].max()

failure_overlap = failure_system20.loc[
    failure_system20['failure_start'].between(usage_min, usage_max, inclusive='both')
].copy()

overlap_summary = pd.DataFrame([
    {'metric': 'System 20 failure rows on nodes 0-255', 'value': len(failure_system20)},
    {'metric': 'Failures inside usage time window', 'value': len(failure_overlap)},
    {'metric': 'Unique failure nodes in overlap window', 'value': failure_overlap['nodenumz'].nunique()},
    {'metric': 'Unique usage nodes with node info', 'value': usage_with_nodes['node'].nunique()},
    {'metric': 'First usage timestamp', 'value': usage_min},
    {'metric': 'Last usage timestamp', 'value': usage_max},
])

display(overlap_summary)

monthly_failure_overlap = failure_overlap.set_index('failure_start').resample('ME').size().rename('failures')
monthly_usage_overlap = usage_df.loc[usage_df['has_node_info']].set_index('usage_time').resample('ME').size().rename('usage_jobs')
trend_compare = pd.concat([monthly_failure_overlap, monthly_usage_overlap], axis=1).fillna(0)
trend_compare['failure_index'] = trend_compare['failures'] / trend_compare['failures'].max()
trend_compare['usage_index'] = trend_compare['usage_jobs'] / trend_compare['usage_jobs'].max()
trend_plot = trend_compare[['failure_index', 'usage_index']].reset_index().melt(id_vars='index', var_name='series', value_name='scaled_count')
trend_plot = trend_plot.rename(columns={'index': 'month'})

plt.figure(figsize=(14, 6))
sns.lineplot(data=trend_plot, x='month', y='scaled_count', hue='series', marker='o')
plt.title('Normalized Monthly Trend: Failure Events vs Usage Jobs')
plt.xlabel('Month')
plt.ylabel('Scaled count')
plt.tight_layout()
plt.show()

## Next Step

Use [02_data_cleaning.ipynb](./02_data_cleaning.ipynb) to load the interim exports, validate the cleaned tables, and prepare the inputs for feature engineering.